In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.action_chains import ActionChains, ScrollOrigin
import pandas as pd
import os
import re
import time
import numpy as np
import traceback

In [2]:
vivino_hrefs_df = pd.read_csv('../data/vivino/filtered_matching_products.csv')
dm_wine_df = pd.read_csv("../data/cleaned_wine_df.csv")

### Get vintage years of dm products

In [3]:
vivino_hrefs_df.head()

,dm_brand_name,dm_product_name,dm_href,vivino_product_name,vivino_href,brand_present,levenshtein_similarity,jaccard_similarity,similarity_difference
0,Tar & Roses,Tempranillo,https://www.danmurphys.com.au/product/DM_91771...,Tar & Roses Tempranillo,https://www.vivino.com/AU/en/wines/4866758,True,1.000000,1.0,0.000000
1,Fraser Gallop Estate,Margaret River Cabernet Merlot,https://www.danmurphys.com.au/product/DM_91766...,Fraser Gallop Estate Cabernet - Merlot,https://www.vivino.com/AU/en/wines/2204702,True,0.466667,0.5,0.033333
2,Akarua Rua,Central Otago Pinot Noir,https://www.danmurphys.com.au/product/DM_91762...,Akarua Rua Pinot Noir,https://www.vivino.com/AU/en/wines/4857731,True,0.416667,0.5,0.083333
3,Paracombe,Adelaide Hills Pinot Noir,https://www.danmurphys.com.au/product/DM_91750...,Paracombe Pinot Noir,https://www.vivino.com/AU/en/wines/4750715,True,0.400000,0.5,0.100000
4,Shingleback,D Block McLaren Vale Cabernet Sauvignon,https://www.danmurphys.com.au/product/DM_91710...,Shingleback D Block Reserve Cabernet Sauvignon,https://www.vivino.com/AU/en/wines/4731697,True,0.769231,0.5,0.269231


In [4]:
dm_wine_df.head()

,href,product_brand,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness,leftover_words,type_as_list
0,https://www.danmurphys.com.au/product/DM_99940...,Sherrah,Red et Al Grenache Shiraz Nero,Bush vine Grenache stars in this original blen...,Red,red blend,2020*,South Australia,McLaren Vale,Alex Sherrah,750mL,29.99,NaN,Medium Bodied,Dry,['other'],['red blend']
1,https://www.danmurphys.com.au/product/DM_99939...,Sherrah,Preservative Free Grenache,"Lifted and Luscious, bright fresh red berries ...",Red,grenache,2021*,South Australia,McLaren Vale,Alex Sherrah,750mL,26.99,NaN,Medium Bodied,Dry,[],['grenache']
2,https://www.danmurphys.com.au/product/DM_99865...,Rabo de Gala,Rose,We welcome a brilliant rosé to our Portuguese ...,Red,cabernet sauvignon,NaN,NaN,NaN,NaN,750mL,11.99,NaN,NaN,NaN,"['castelao', 'and']",['cabernet sauvignon']
3,https://www.danmurphys.com.au/product/DM_99831...,Vincent's Breeze,McLaren Vale Shiraz 2019,Vincent’s Breeze wines take their name from a ...,Red,shiraz,2019,South Australia,McLaren Vale,NaN,750mL,24.99,NaN,Full Bodied,NaN,[],['shiraz']
4,https://www.danmurphys.com.au/product/DM_98985...,Thymiopoulos,'Young Vines' Xinomavro,Elegant character of fresh black fruit and her...,Red,NaN,Non Vintage,NaN,Macedonia,Apostolos Thymiopoulos,750ML,31.99,NaN,Full Bodied,NaN,['xinomavro'],[]


In [5]:
vivino_hrefs_df = vivino_hrefs_df.merge(dm_wine_df, how='left', left_on='dm_href', right_on='href')
vivino_hrefs_df = vivino_hrefs_df[["dm_brand_name", "dm_product_name", "dm_href", "vivino_product_name", "vivino_href", "vintage"]]
vivino_hrefs_df.head()

,dm_brand_name,dm_product_name,dm_href,vivino_product_name,vivino_href,vintage
0,Tar & Roses,Tempranillo,https://www.danmurphys.com.au/product/DM_91771...,Tar & Roses Tempranillo,https://www.vivino.com/AU/en/wines/4866758,2020*
1,Fraser Gallop Estate,Margaret River Cabernet Merlot,https://www.danmurphys.com.au/product/DM_91766...,Fraser Gallop Estate Cabernet - Merlot,https://www.vivino.com/AU/en/wines/2204702,2019
2,Akarua Rua,Central Otago Pinot Noir,https://www.danmurphys.com.au/product/DM_91762...,Akarua Rua Pinot Noir,https://www.vivino.com/AU/en/wines/4857731,2021
3,Paracombe,Adelaide Hills Pinot Noir,https://www.danmurphys.com.au/product/DM_91750...,Paracombe Pinot Noir,https://www.vivino.com/AU/en/wines/4750715,2021
4,Shingleback,D Block McLaren Vale Cabernet Sauvignon,https://www.danmurphys.com.au/product/DM_91710...,Shingleback D Block Reserve Cabernet Sauvignon,https://www.vivino.com/AU/en/wines/4731697,2018*


In [6]:
vivino_hrefs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3826 entries, 0 to 3825
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   dm_brand_name        3826 non-null   object
 1   dm_product_name      3826 non-null   object
 2   dm_href              3826 non-null   object
 3   vivino_product_name  3826 non-null   object
 4   vivino_href          3826 non-null   object
 5   vintage              3348 non-null   object
dtypes: object(6)
memory usage: 179.5+ KB


In [7]:
vivino_hrefs_df[vivino_hrefs_df.isna().any(axis=1)]

,dm_brand_name,dm_product_name,dm_href,vivino_product_name,vivino_href,vintage
33,Just Like Magic,Pinot Noir,https://www.danmurphys.com.au/product/DM_20021...,Just Like Magic Pinot Noir,https://www.vivino.com/AU/en/wines/172832070,NaN
84,Jacob's Creek,Reserve Cabernet Sauvignon,https://www.danmurphys.com.au/product/DM_13271...,Jacob'S Creek Reserve Cabernet Sauvignon,https://www.vivino.com/AU/en/wines/2313385,NaN
112,The Painter,Shiraz,https://www.danmurphys.com.au/product/DM_12010...,The Painter Shiraz,https://www.vivino.com/AU/en/wines/168047237,NaN
126,Shingleback,Red Knot Cabernet Sauvignon,https://www.danmurphys.com.au/product/DM_11164...,Shingleback Red Knot Cabernet Sauvignon,https://www.vivino.com/AU/en/wines/2764599,NaN
142,Lindeman's,Early Harvest Semillon Sauvignon Blanc,https://www.danmurphys.com.au/product/DM_90601...,Lindeman'S Early Harvest Sémillon - Sauvignon ...,https://www.vivino.com/AU/en/wines/1769638,NaN
...,...,...,...,...,...,...
3806,Tread Softly,Grenache,https://www.danmurphys.com.au/product/DM_87693...,Tread Softly Grenache,https://www.vivino.com/AU/en/wines/164637840,NaN
3810,Sit Stay Society,Max's Cabernet Malbec 375mL,https://www.danmurphys.com.au/product/DM_87541...,Sit Stay Society Max'S Cabernet - Malbec,https://www.vivino.com/AU/en/wines/160998613,NaN
3811,Sit Stay Society,Toby's Shiraz Tempranillo 375mL,https://www.danmurphys.com.au/product/DM_87541...,Sit Stay Society Toby'S Shiraz - Tempranillo,https://www.vivino.com/AU/en/wines/160896903,NaN
3813,Tread Softly,Pinot Noir 375mL,https://www.danmurphys.com.au/product/DM_87407...,Tread Softly Pinot Noir,https://www.vivino.com/AU/en/wines/160740250,NaN


In [8]:
vivino_hrefs_df.vintage.value_counts().index

Index(['2021', '2019', '2020', '2018', 'Non Vintage', '2021*', '2022', '2017',
       '2020*', '2022*', '2016', '2019*', '2015', '2018*', '2014', '2013',
       '2023*', '2012', '2023', '2017*', '2010', 'Multiple(Bundles)',
       'Non Vintage*', '2016*', '2015*', '1998', '2009', '2006', '2011',
       '2008', '2014*', '2005', '2004', '2002', '1996', '2001', '2012*',
       '1955', '1999', 'Cork', '1990', '2009*', '1991', '1993', '1997',
       'Non VIntage', '1967', '2013*', 'NV', '2020/2021', '1994', '2011*',
       'Various', '1985', '1974', '1981', '1976', '2003', '1958', '1954',
       '1965', '1964', '1953', '1983', 'Screw Cap', '2019; 2021', '1961',
       '1963', '1962', '2008*', '1989', '2024', '1986', '2010*', '2007'],
      dtype='object', name='vintage')

In [9]:
vivino_hrefs_df[vivino_hrefs_df["vintage"] == "2020/2021"]

,dm_brand_name,dm_product_name,dm_href,vivino_product_name,vivino_href,vintage
650,Patient Cottat,Sancerre 'Anciennes Vignes',https://www.danmurphys.com.au/product/DM_20220...,Patient Cottat Sancerre Anciennes Vignes,https://www.vivino.com/AU/en/wines/2600118,2020/2021
652,Famille Bougrier,Confidences Vouvray,https://www.danmurphys.com.au/product/DM_20219...,Famille Bougrier Confidences Vouvray Sec,https://www.vivino.com/AU/en/wines/166046575,2020/2021


In [10]:
def extract_years(value):
    # Find all 4-digit year patterns in the string
    try:
        years = re.findall(r'\b\d{4}\b', value)
        return years if years else []
    except TypeError:
        return []

# Apply the extraction function to the 'vintage' column
vivino_hrefs_df['extracted_years'] = vivino_hrefs_df['vintage'].apply(extract_years)
vivino_hrefs_df

,dm_brand_name,dm_product_name,dm_href,vivino_product_name,vivino_href,vintage,extracted_years
0,Tar & Roses,Tempranillo,https://www.danmurphys.com.au/product/DM_91771...,Tar & Roses Tempranillo,https://www.vivino.com/AU/en/wines/4866758,2020*,[2020]
1,Fraser Gallop Estate,Margaret River Cabernet Merlot,https://www.danmurphys.com.au/product/DM_91766...,Fraser Gallop Estate Cabernet - Merlot,https://www.vivino.com/AU/en/wines/2204702,2019,[2019]
2,Akarua Rua,Central Otago Pinot Noir,https://www.danmurphys.com.au/product/DM_91762...,Akarua Rua Pinot Noir,https://www.vivino.com/AU/en/wines/4857731,2021,[2021]
3,Paracombe,Adelaide Hills Pinot Noir,https://www.danmurphys.com.au/product/DM_91750...,Paracombe Pinot Noir,https://www.vivino.com/AU/en/wines/4750715,2021,[2021]
4,Shingleback,D Block McLaren Vale Cabernet Sauvignon,https://www.danmurphys.com.au/product/DM_91710...,Shingleback D Block Reserve Cabernet Sauvignon,https://www.vivino.com/AU/en/wines/4731697,2018*,[2018]
...,...,...,...,...,...,...,...
3821,Elephant In The Room,Tempranillo,https://www.danmurphys.com.au/product/DM_86526...,Elephant In The Room Titanic Tempranillo,https://www.vivino.com/AU/en/wines/165182546,2019*,[2019]
3822,Brown Brothers,Winemakers Series Tempranillo Graciano,https://www.danmurphys.com.au/product/DM_86491...,Brown Brothers Winemaker’S Series Tempranillo ...,https://www.vivino.com/AU/en/wines/165373677,2019,[2019]
3823,Noble Fellows,Colonel Kiwi Marlborough Pinot Noir,https://www.danmurphys.com.au/product/DM_86484...,Noble Fellows Colonel Kiwi Pinot Noir,https://www.vivino.com/AU/en/wines/161072244,2020,[2020]
3824,Distant South,Tasmania Pinot Noir,https://www.danmurphys.com.au/product/DM_86462...,Distant South Pinot Noir,https://www.vivino.com/AU/en/wines/160780353,2020*,[2020]


### Start scraping

In [11]:
options = Options()

options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-infobars')
options.add_argument('--disable-extensions')
options.add_argument('--disable-notifications')
options.add_argument('--window-size=1724,968')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"
options.add_argument(f'user-agent={user_agent}')

In [12]:
vivino_hrefs_df

,dm_brand_name,dm_product_name,dm_href,vivino_product_name,vivino_href,vintage,extracted_years
0,Tar & Roses,Tempranillo,https://www.danmurphys.com.au/product/DM_91771...,Tar & Roses Tempranillo,https://www.vivino.com/AU/en/wines/4866758,2020*,[2020]
1,Fraser Gallop Estate,Margaret River Cabernet Merlot,https://www.danmurphys.com.au/product/DM_91766...,Fraser Gallop Estate Cabernet - Merlot,https://www.vivino.com/AU/en/wines/2204702,2019,[2019]
2,Akarua Rua,Central Otago Pinot Noir,https://www.danmurphys.com.au/product/DM_91762...,Akarua Rua Pinot Noir,https://www.vivino.com/AU/en/wines/4857731,2021,[2021]
3,Paracombe,Adelaide Hills Pinot Noir,https://www.danmurphys.com.au/product/DM_91750...,Paracombe Pinot Noir,https://www.vivino.com/AU/en/wines/4750715,2021,[2021]
4,Shingleback,D Block McLaren Vale Cabernet Sauvignon,https://www.danmurphys.com.au/product/DM_91710...,Shingleback D Block Reserve Cabernet Sauvignon,https://www.vivino.com/AU/en/wines/4731697,2018*,[2018]
...,...,...,...,...,...,...,...
3821,Elephant In The Room,Tempranillo,https://www.danmurphys.com.au/product/DM_86526...,Elephant In The Room Titanic Tempranillo,https://www.vivino.com/AU/en/wines/165182546,2019*,[2019]
3822,Brown Brothers,Winemakers Series Tempranillo Graciano,https://www.danmurphys.com.au/product/DM_86491...,Brown Brothers Winemaker’S Series Tempranillo ...,https://www.vivino.com/AU/en/wines/165373677,2019,[2019]
3823,Noble Fellows,Colonel Kiwi Marlborough Pinot Noir,https://www.danmurphys.com.au/product/DM_86484...,Noble Fellows Colonel Kiwi Pinot Noir,https://www.vivino.com/AU/en/wines/161072244,2020,[2020]
3824,Distant South,Tasmania Pinot Noir,https://www.danmurphys.com.au/product/DM_86462...,Distant South Pinot Noir,https://www.vivino.com/AU/en/wines/160780353,2020*,[2020]


# Going to Scrape:
### username
### user href
### rating
### date
### vintage (if present)

In [ ]:

def scroll_to(element):
    ActionChains(driver)\
        .scroll_to_element(element)\
        .perform()
    
def scroll_from(element, amount):
    scroll_origin = ScrollOrigin.from_element(element)
    ActionChains(driver)\
        .scroll_from_origin(scroll_origin, 0, amount)\
        .perform()

# setup driver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# lists to make into dataframe
dm_hrefs_subset = []
dm_brand_name_subsets = []
dm_product_name_subsets = []
dm_vintage_years_subset = []
vivino_hrefs_subset = []
all_reviews_subset = []

total_wines = len(vivino_hrefs_df)
#total_wines = 2 # Used for testing
starting_i = 826
#n_waits_limit = 3
#n_rescrolls_limit = 5
timeout_duration = 10  # Set the timeout duration
finished = False

class TooManyChecksException(Exception):
    def __init__(self, message):
        super().__init__(message)

def scrape_and_append_review_cards(driver, all_product_reviews, last_scraped_j):
    # scrape review card data
    reviews_window = driver.find_element(By.CLASS_NAME, "baseModal__window--3r5PC")
    review_cards = reviews_window.find_elements(By.CLASS_NAME, "communityReviewItem__reviewCard--1RupJ")
    print("Scraping review cards...")
    #for review_card in review_cards:
    for j in range(last_scraped_j, len(review_cards), 1):
        #n_review_cards = len(review_cards)
        #print(f"Scraping review card j={j} of {n_review_cards-1}...")
        print(f"Scraping review card i={j}...")
        review_card = review_cards[j]
        n_loops_limit = 5
        n_loops = 0
        while True:
            try:
                rating = review_card.find_element(By.CLASS_NAME, "userRating_ratingNumber__cMtKU").text
                break
            except StaleElementReferenceException:
                if n_loops < n_loops_limit:
                    # Deal with review cards not being fully loaded
                    #print(f"Review card j={j} was not fully loaded upon retrieval of review cards. Retrying...")
                    print(f"Review card i={j} was not fully loaded upon retrieval of review cards. Retrying...")
                    reviews_window = driver.find_element(By.CLASS_NAME, "baseModal__window--3r5PC")
                    review_cards = reviews_window.find_elements(By.CLASS_NAME, "communityReviewItem__reviewCard--1RupJ")
                    review_card = review_cards[j]
                    #print(f"{n_review_cards} should equal {len(review_cards)}")
                    n_loops += 1
                    time.sleep(0.5)
                    continue
                else:
                    raise TooManyChecksException(f"Tried and failed to retrieve review card j={j} too many times due to StaleElementReferenceException") 
        user_href = ""
        username = ""
        user_review_date = ""
        # check for vintage year of review
        vintage_year = ""
        try:
            # Try to find vintage year
            vintage_year = review_card.find_element(By.CLASS_NAME, "reviewedVintageYear__vintageText--3TZOW").text
        except NoSuchElementException:
            pass
        # loop through anchors to get username and date of review
        user_anchors = review_card.find_elements(By.CLASS_NAME, "anchor_anchor__m8Qi-")
        for anchor in user_anchors:
            if "userAlias_userAlias__ztmrT" in anchor.get_attribute("class"):
                username = anchor.text
                user_href = anchor.get_attribute('href')
            if "reviewDate__reviewDate--49vpM" in anchor.get_attribute("class"):
                user_review_date = anchor.text
    
        user_review = {
            "href" : user_href,
            "username" : username,
            "rating" : rating,
            "date" : user_review_date,
            "vintage_year" : vintage_year
            }
        all_product_reviews.append(user_review)
        last_scraped_j = j
    return all_product_reviews, last_scraped_j

while starting_i < (total_wines -1):
    # handles unexpected exceptions and save progress
    try:
        for i in range(starting_i, total_wines, 1):
        
            row = vivino_hrefs_df.iloc[i]
            href = row.vivino_href
            print(f"\n\nGetting product reviews for i={i} of {total_wines-1}: {href}")
            
            # Set the page load timeout
            driver.set_page_load_timeout(timeout_duration)
            
            # load product page
            while True:
                try:
                    # Try to load the page
                    driver.get(href)
                    break
                except TimeoutException:
                    # Handle the timeout exception
                    print(f"The page took longer than {timeout_duration} seconds to load and was aborted. Trying again...")
                    continue

            # show all reviews
            #######
            all_reviws_anchor = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.XPATH, "//*[contains(@class, 'anchor_anchor__m8Qi-') and contains(@class, 'scroll')]"))
                )
            print("Found reviews anchor")
            anchor_found = True
            previous_page_text = driver.find_element
            all_reviws_anchor.click()   
            """
            time.sleep(1)
            anchor_found = False
            anchors = driver.find_elements(By.CLASS_NAME, "anchor_anchor__m8Qi-")
            for anchor in anchors:
                if "scroll" in anchor.get_attribute("class"):
                    print("Found reviews anchor")
                    all_reviws_anchor = anchor
                    anchor_found = True
                    previous_page_text = driver.find_element
                    all_reviws_anchor.click()
                    break
            if not anchor_found:
                raise NoSuchElementException('Could not find reviews anchor')
            """
            ######

            # Click show more reviews
            #time.sleep(1)
            reviews_found = False
            reviews_button_loaded = False
            n_waits = 0
            n_wait_limit = 5
            while not reviews_button_loaded:
                buttons = driver.find_elements(By.CLASS_NAME, "MuiButtonBase-root")
                for button in buttons:
                    if button.text == "Show more reviews":
                        show_more_reviews_button = button
                        print("Found show more reviews button. Clicking...")
                        reviews_button_loaded = True
                        reviews_found = True
                        button.click()
                if not reviews_button_loaded:
                    if n_waits < n_wait_limit:
                        print("Show more reviews button not loaded. Waiting and trying again...")
                        time.sleep(0.5)
                        n_waits += 1
                    else:
                        print("Wait limit for button exceeded. No Reviews Found.")
                        reviews_button_loaded = True # Set to true if there is no button to load

            all_product_reviews = []
            if reviews_found:
                # Click sort by most_recent
                time.sleep(1)
                sort_recent_anchor = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.XPATH, "//a[contains(@class, 'nchor_anchor__m8Qi-') and contains(text(), 'Recent')]"))
                )
                sort_recent_anchor.click()

                # Scroll down to load all reviews
                n_waits = 0
                n_waits_limit = 3
                n_rescrolls_limit = 5
                #reviews_window = driver.find_element(By.CLASS_NAME, "baseModal__window--3r5PC")
                reviews_window = WebDriverWait(driver, 10).until(
                                EC.presence_of_element_located((By.CLASS_NAME, "baseModal__window--3r5PC"))
                            )
                review_cards = reviews_window.find_elements(By.CLASS_NAME, "communityReviewItem__reviewCard--1RupJ")
                n_review_cards = len(review_cards)
                previous_n_review_cards = n_review_cards
                all_reviews_loaded = False


                try:
                    scrape_limit = 200
                    last_scraped_j = 0
                    while not all_reviews_loaded and last_scraped_j < 200:
                        print("number of loaded review cards =", n_review_cards)
                        reviews_window = driver.find_element(By.CLASS_NAME, "baseModal__window--3r5PC")
                        review_cards = reviews_window.find_elements(By.CLASS_NAME, "communityReviewItem__reviewCard--1RupJ")
                        last_review_card = review_cards[-1]
                        print("Scrolling to last review...")
                        scroll_to(last_review_card)
                        #loop until new reviews load
                        n_waits = 0
                        n_rescrolls = 0
                        while True:
                            time.sleep(1)
                            reviews_window = driver.find_element(By.CLASS_NAME, "baseModal__window--3r5PC")
                            review_cards = reviews_window.find_elements(By.CLASS_NAME, "communityReviewItem__reviewCard--1RupJ")
                            n_review_cards = len(review_cards)
                            print(f"Comparing number of current reviews: {n_review_cards} to number of previous reviews: {previous_n_review_cards}")
                            if n_review_cards == previous_n_review_cards:
                                if n_waits <= n_waits_limit:
                                    print("Review cards have not updated. Waiting...")
                                    #time.sleep(1)
                                    n_waits += 1
                                    continue
                                elif n_rescrolls < n_rescrolls_limit:
                                    time.sleep(1)
                                    last_review_card = review_cards[-1]
                                    # scroll up and down to try refreshing page
                                    print("Attempting to rescroll page to load more reviews...")
                                    scroll_from(last_review_card, -500)
                                    time.sleep(1)
                                    scroll_to(last_review_card)
                                    time.sleep(0.2)
                                    scroll_from(last_review_card, 100)
                                    n_rescrolls += 1
                                    n_waits = 0
                                    continue
                                else:
                                    print("No more review cards to load. Stopping scrolling...")
                                    all_reviews_loaded = True
                                    if i == (total_wines-1):
                                        finished = True
                                        print("\n#############!!!ALL PRODUCTS COMPLETE!!!#############\n")
                                    break
                            else:
                                previous_n_review_cards = n_review_cards
                                n_rescrolls = 0
                                # Scrape review cards
                                time.sleep(1)
                                all_product_reviews, last_scraped_j = scrape_and_append_review_cards(driver, all_product_reviews, last_scraped_j+1)
                                if last_scraped_j > scrape_limit:
                                    print(f"Number of scraped reviews has exceeded limit of {scrape_limit} for this product. Saving and continuing to next product...")
                                break
                        time.sleep(2)
                except TimeoutException:
                    print("\n#############!!!TimeOutException BEFORE GETTING ALL REVIEWS!!!#############")
                    print(f"Scraped {last_scraped_j} review cards before timeout. Saving and moving to next product\n")

            # append to data lists
            dm_hrefs_subset.append(row.dm_href)
            dm_vintage_years_subset.append(row.extracted_years)
            dm_brand_name_subsets.append(row.dm_brand_name)
            dm_product_name_subsets.append(row.dm_product_name)
            vivino_hrefs_subset.append(row.vivino_href)
            all_reviews_subset.append(all_product_reviews)
            #break # break after first loop while developing
    
    except Exception as e:
        print(e)
        print(f"Stopped scraping at i ={i} of {total_wines-1}: {href}")
    
    finally:

        #save results
        vivino_reviews_df = {
            "dm_href" : dm_hrefs_subset,
            "dm_vintage_years": dm_vintage_years_subset,
            "dm_brand_name": dm_brand_name_subsets,
            "dm_product_name": dm_product_name_subsets,
            "vivino_href" : vivino_hrefs_subset,
            "all_reviews" : all_reviews_subset,
        }
        vivino_reviews_df = pd.DataFrame(vivino_reviews_df)
    
        # save df to csv
        if finished:
            file_path = f"../data/vivino/user_resviews_subsets/vivino_user_reviews_dataset_i_{starting_i}-{i}.csv"
        else:
            file_path = f"../data/vivino/user_resviews_subsets/vivino_user_reviews_dataset_i_{starting_i}-{i-1}.csv"
        print(f"Saving results to {file_path}...")
        vivino_reviews_df.to_csv(file_path, index=False)
    
        starting_i = i
    
        time.sleep(2)
    
# Close the webpage
driver.quit()




Getting product reviews for i=793 of 3825: https://www.vivino.com/AU/en/wines/171369852
The page took longer than 10 seconds to load and was aborted. Trying again...
Found reviews anchor
Show more reviews button not loaded. Waiting and trying again...
Show more reviews button not loaded. Waiting and trying again...
Found show more reviews button. Clicking...
number of loaded review cards = 13
Scrolling to last review...
Comparing number of current reviews: 13 to number of previous reviews: 13
Review cards have not updated. Waiting...
Comparing number of current reviews: 20 to number of previous reviews: 13
Scraping review cards...
Scraping review card i=1...
Scraping review card i=2...
Scraping review card i=3...
Scraping review card i=4...
Scraping review card i=5...
Scraping review card i=6...
Scraping review card i=7...
Scraping review card i=8...
Scraping review card i=9...
Scraping review card i=10...
Scraping review card i=11...
Scraping review card i=12...
Scraping review card

In [ ]:
"""
def scroll_to(element):
    ActionChains(driver)\
        .scroll_to_element(eleme
nt)\
        .perform()
    
def scroll_from(element, amount):
    scroll_origin = ScrollOrigin.from_element(element)
    ActionChains(driver)\
        .scroll_from_origin(scroll_origin, 0, amount)\
        .perform()

# setup driver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# lists to make into dataframe
dm_hrefs_subset = []
dm_brand_name_subsets = []
dm_product_name_subsets = []
dm_vintage_years_subset = []
vivino_hrefs_subset = []
all_reviews_subset = []

total_wines = len(vivino_hrefs_df)
#total_wines = 2 # Used for testing
starting_i = 0
#n_waits_limit = 3
#n_rescrolls_limit = 5
timeout_duration = 10  # Set the timeout duration
finished = False

while starting_i < (total_wines -1):
    # handles unexpected exceptions and save progress
    try:
        for i in range(starting_i, total_wines, 1):
        
            row = vivino_hrefs_df.iloc[i]
            href = row.vivino_href
            print(f"\n\nGetting product reviews for i={i} of {total_wines-1}: {href}")
            
            # Set the page load timeout
            driver.set_page_load_timeout(timeout_duration)
            
            # load product page
            while True:
                try:
                    # Try to load the page
                    driver.get(href)
                    break
                except TimeoutException:
                    # Handle the timeout exception
                    print(f"The page took longer than {timeout_duration} seconds to load and was aborted. Trying again...")
                    continue

            # show all reviews
            anchor_found = False
            anchors = driver.find_elements(By.CLASS_NAME, "anchor_anchor__m8Qi-")
            for anchor in anchors:
                if "scroll" in anchor.get_attribute("class"):
                    print("Found reviews anchor")
                    all_reviws_anchor = anchor
                    anchor_found = True
                    previous_page_text = driver.find_element
                    all_reviws_anchor.click()
                    break
            if not anchor_found:
                raise NoSuchElementException('Could not find reviews anchor')
        
            # Click show more reviews
            reviews_found = False
            reviews_button_loaded = False
            n_waits = 0
            n_wait_limit = 5
            while not reviews_button_loaded:
                buttons = driver.find_elements(By.CLASS_NAME, "MuiButtonBase-root")
                for button in buttons:
                    if button.text == "Show more reviews":
                        show_more_reviews_button = button
                        print("Found show more reviews button. Clincking...")
                        reviews_button_loaded = True
                        reviews_found = True
                        button.click()
                if not reviews_button_loaded:
                    if n_waits < n_wait_limit:
                        print("Show more reviews button not loaded. Waiting and trying again...")
                        time.sleep(0.5)
                        n_waits += 1
                    else:
                        print("Wait limit for button exceeded. No Reviews Found.")
                        reviews_button_loaded = True # Set to true if there is no button to load

            # Click sort by most_recent
            #time.sleep(1)
            sort_recent_anchor = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, "//a[contains(@class, 'nchor_anchor__m8Qi-') and contains(text(), 'Recent')]"))
            )
            sort_recent_anchor.click()
                        
            all_product_reviews = []
            if reviews_found:
                # Scroll down to load all reviews
                n_waits = 0
                n_waits_limit = 3
                n_rescrolls_limit = 5
                reviews_window = driver.find_element(By.CLASS_NAME, "baseModal__window--3r5PC")
                review_cards = reviews_window.find_elements(By.CLASS_NAME, "communityReviewItem__reviewCard--1RupJ")
                n_review_cards = len(review_cards)
                previous_n_review_cards = n_review_cards
                all_reviews_loaded = False
                n_reviews_limit = 200
                n_reviews = 0
                while not all_reviews_loaded and n_reviews < n_reviews_limit(:
                    print("number of loaded review cards =", n_review_cards)
                    reviews_window = driver.find_element(By.CLASS_NAME, "baseModal__window--3r5PC")
                    review_cards = reviews_window.find_elements(By.CLASS_NAME, "communityReviewItem__reviewCard--1RupJ")
                    last_review_card = review_cards[-1]
                    print("Scrolling to last review...")
                    scroll_to(last_review_card)
                    #loop until new reviews load
                    n_waits = 0
                    n_rescrolls = 0
                    while True:
                        reviews_window = driver.find_element(By.CLASS_NAME, "baseModal__window--3r5PC")
                        review_cards = reviews_window.find_elements(By.CLASS_NAME, "communityReviewItem__reviewCard--1RupJ")
                        n_review_cards = len(review_cards)
                        print(f"Comparing number of current reviews: {n_review_cards} to number of previous number of reviews: {previous_n_review_cards}")
                        if n_review_cards == previous_n_review_cards:
                            if n_waits <= n_waits_limit:
                                print("Review cards have not updated. Waiting...")
                                time.sleep(1)
                                n_waits += 1
                                continue
                            elif n_rescrolls < n_rescrolls_limit:
                                # scroll up and down to try refreshing page
                                print("Attempting to rescroll page to load more reviews...")
                                scroll_from(last_review_card, -500)
                                time.sleep(1)
                                scroll_to(last_review_card)
                                time.sleep(0.2)
                                scroll_from(last_review_card, 100)
                                n_rescrolls += 1
                                n_waits = 0
                                continue
                            else:
                                print("No more review cards to load. Stopping scrolling...")
                                all_reviews_loaded = True
                                if i == (total_wines-1):
                                    finished = True
                                    print("\n#############!!!ALL PRODUCTS COMPLETE!!!#############\n")
                                break
                        else:
                            previous_n_review_cards = n_review_cards
                            n_rescrolls = 0
                            break
                    time.sleep(2)
                
                # scrape review card data
                reviews_window = driver.find_element(By.CLASS_NAME, "baseModal__window--3r5PC")
                review_cards = reviews_window.find_elements(By.CLASS_NAME, "communityReviewItem__reviewCard--1RupJ")
                print("Scraping review cards...")
                #for review_card in review_cards:
                for j in range(len(review_cards)):
                    n_review_cards = len(review_cards)
                    print(f"Scraping review card j={j} of {n_review_cards-1}...")
                    review_card = review_cards[j]
                    rating = review_card.find_element(By.CLASS_NAME, "userRating_ratingNumber__cMtKU").text
                    user_href = ""
                    username = ""
                    user_review_date = ""
                    # check for vintage year of review
                    vintage_year = ""
                    try:
                        # Try to find vintage year
                        vintage_year = review_card.find_element(By.CLASS_NAME, "reviewedVintageYear__vintageText--3TZOW").text
                    except NoSuchElementException:
                        pass
                    # loop through anchors to get username and date of review
                    user_anchors = review_card.find_elements(By.CLASS_NAME, "anchor_anchor__m8Qi-")
                    for anchor in user_anchors:
                        if "userAlias_userAlias__ztmrT" in anchor.get_attribute("class"):
                            username = anchor.text
                            user_href = anchor.get_attribute('href')
                        if "reviewDate__reviewDate--49vpM" in anchor.get_attribute("class"):
                            user_review_date = anchor.text
                
                    user_review = {
                        "href" : user_href,
                        "username" : username,
                        "rating" : rating,
                        "date" : user_review_date,
                        "vintage_year" : vintage_year
                        }
                    all_product_reviews.append(user_review)

            # append to data lists
            dm_hrefs_subset.append(row.dm_href)
            dm_vintage_years_subset.append(row.extracted_years)
            dm_brand_name_subsets.append(row.dm_brand_name)
            dm_product_name_subsets.append(row.dm_product_name)
            vivino_hrefs_subset.append(row.vivino_href)
            all_reviews_subset.append(all_product_reviews)
            #break # break after first loop while developing
    
    except Exception as e:
        print(e)
        print(f"Stopped scraping at i ={i} of {total_wines-1}: {href}")
    
    finally:

        #save results
        vivino_reviews_df = {
            "dm_href" : dm_hrefs_subset,
            "dm_vintage_years": dm_vintage_years_subset,
            "dm_brand_name": dm_brand_name_subsets,
            "dm_product_name": dm_product_name_subsets,
            "vivino_href" : vivino_hrefs_subset,
            "all_reviews" : all_reviews_subset,
        }
        vivino_reviews_df = pd.DataFrame(vivino_reviews_df)
    
        # save df to csv
        if finished:
            file_path = f"../data/vivino/user_resviews_subsets/vivino_user_reviews_dataset_i_{starting_i}-{i}.csv"
        else:
            file_path = f"../data/vivino/user_resviews_subsets/vivino_user_reviews_dataset_i_{starting_i}-{i-1}.csv"
        print(f"Saving results to {file_path}...")
        vivino_reviews_df.to_csv(file_path, index=False)
    
        starting_i = i
    
        time.sleep(2)
    
# Close the webpage
driver.quit()
"""

In [ ]:
"""
# Click sort by most_recent
#time.sleep(1)
sort_recent_anchor = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.XPATH, "//a[contains(@class, 'nchor_anchor__m8Qi-') and contains(text(), 'Recent')]"))
)
sort_recent_anchor.click()
"""